# Design of Welded Structures - Implementation Notes

Original Python implementations of section property calculations and
built-up cross-section analysis. Concepts and problem definitions
reference:

- Blodgett, O.W. *Design of Welded Structures*. Lincoln Electric.
- Miller, D. *Welded Connections -- A Primer for Engineers*, AISC Design Guide 21.

Refer to the original texts for theoretical background and derivations.


# 1 - Introduction

## 1.1 - Introduction to Welded Construction

<div class="page-break"></div>

# 2 - Load & Stress Analysis

## 2.1 - Properties of Materials

## 2.2 - Properties of Sections

In [1]:
n = ((4*6*14 + 2*12*6 + 4*8*2) / (4*6 + 2*12 + 4*8))
n

In [2]:
# Section property functions are now available in civilpy.structural.section_properties
from civilpy.structural.section_properties import (
    get_rectangular_section_properties,
    get_rectangular_section_properties_baseline,
    get_triangular_section_properties,
    get_triangular_section_properties_baseline,
    get_bar_section_properties,
    get_pipe_section_properties,
    get_oval_section_properties,
    get_hollow_oval_section_properties,
)
import math


In [3]:
def get_rectangular_section_properties(b, d):  # b = width, d = height
    """
    Takes the rectangle's width (b) and it's height (d) and returns the Moment 
    of inertia (I), the Section Modulus (S) and the Radius of Gyration (r).  
    All properties are calculated with respect to the Neutral Axis.
    """
    I = (b * d ** 3) / 12
    S = (b * d ** 2) / 6
    r = d / math.sqrt(12)

    return I, S, r

In [4]:
def get_rectangular_section_properties_baseline(b, d):
    """
    Takes the rectangle's width (b) and it's height (d) and returns the Moment 
    of inertia (I), the Section Modulus (S) and the Radius of Gyration (r).  
    All properties are calculated with respect to the baseline of the 
    rectangle.
    """
    I = (b * d ** 3) / 3
    S = (b * d ** 2) / 3
    r = d / math.sqrt(13)

    return I, S, r

In [5]:
def get_triangular_section_properties(b, d):
    """
    Takes the triangle's width (b) and it's height (d) and returns the Moment 
    of inertia (I), the Section Modulus (S) and the Radius of Gyration (r).  
    All properties are calculated with respect to the Neutral Axis of the 
    triangle.
    """
    I = (b * d ** 3) / 36
    S = (b * d ** 2) / 24
    r = d / math.sqrt(18)

    return I, S, r

In [6]:
def get_triangular_section_properties_baseline(b, d):
    """
    Takes the triangle's width (b) and it's height (d) and returns the Moment 
    of inertia (I), the Section Modulus (S) and the Radius of Gyration (r).  
    All properties are calculated with respect to the baseline of the triangle.
    """
    I = (b * d ** 3) / 12
    S = (b * d ** 2) / 12
    r = d / math.sqrt(6)

    return I, S, r

In [7]:
def get_bar_section_properties(d):
    """
    Takes the bar section's diameter (d) and returns the Moment of inertia (I),
    the Section Modulus (S) and the Radius of Gyration (r).  All properties are 
    calculated with respect to the neutral axis of the bar (centerline).
    """
    I = (math.pi * d ** 4) / 64
    S = (math.pi * d ** 3) / 32
    r = d / 4

    return I, S, r

In [8]:
def get_pipe_section_properties(D, d):
    """
    Takes the Pipe section's outer diameter (D) and it's inner diameter (d) and 
    returns the Moment of inertia (I), the Section Modulus (S) and the Radius 
    of Gyration (r).  All properties are calculated with respect to the Neutral 
    axis of the pipe (centerline).
    """
    I = (math.pi / 64) * (D**4 - d**4)
    S = (math.pi / 32) * (D**4 - d**4) / D
    r = math.sqrt(D**2 + d**2) / 4

    return I, S, r

In [9]:
def get_oval_section_properties(b, a):
    """
    Takes the oval's height (a) and it's width (b) and returns the Moment of 
    inertia (I), the Section Modulus (S) and the Radius of Gyration (r).  All 
    properties are calculated with respect to the neutral axis of the oval 
    (centerline).
    """
    I = (math.pi * a**3 * b) / 4
    S = (math.pi * a**2 * b) / 4
    r = a / 2

    return I, S, r

In [10]:
def get_hollow_oval_section_properties(a, b, c, d):
    """
    Takes the oval's I.D. heihgt (c), I.D. width (d), O.D. width (b) and it's 
    O.D. height (a) and returns the Moment of inertia (I), the Section Modulus 
    (S) and the Radius of Gyration (r).  All properties are calculated with 
    respect to the neutral axis of the rectangle (centerline).
    """
    I = (math.pi / 4) * (a**3*b - c**3*d)
    S = (math.pi * (a**3*b - c**3*d)) / (4*a)
    r = (1/a) * math.sqrt((a**3*b-c**3-d) / (a*b-c*d))

    return I, S, r

In [ ]:
I, _, _ = get_rectangular_section_properties(6, 4)
I

In [12]:
# Using python for the above formula
# Independent moment of inertias
top_flange_I, _, _ = get_rectangular_section_properties(6, 4)
web_I, _, _ = get_rectangular_section_properties(2, 8)
bot_flange_I, _, _ = get_rectangular_section_properties(10, 4)

# Areas
top_flange_A = 6 * 4
web_A = 2 * 8
bot_flange_A = 10 * 4

# Distances to the Neutral axis
tf_na = 7.2
web_na = 1.2
bf_na = 4.8

print(int(top_flange_I + top_flange_A * tf_na ** 2 + 
      web_I + web_A * web_na ** 2 +
      bot_flange_I + bot_flange_A * bf_na ** 2))

In [13]:
import pandas as pd
# See note - Dataframes aren't necessary, only used to match book example

data = {
    'Plate': ['A', 'B', 'C'],
    'Size': [(10, 4), (2, 8), (6, 4)],  # Note the use of tuples for dims
    'Distance $y$': ['', '', ''],
    '$A=b*d$': ['', '', ''],
    '$M=A*y$': ['', '', ''],
    '$I_y=Ay^2$': ['', '', ''],
    '$I_g=\\frac{bd^3}{12}$': ['', '', '']
}

df = pd.DataFrame(data)
df

In [14]:
# Calculating the 'y' column
height = 0

for i, plate in df.iterrows():         # Iterate through the table by row
    height += plate.Size[1] / 2        # Calculate the y value for each plate
    df.at[i, 'Distance $y$'] = height  # Stores the y value in the table
    height += plate.Size[1] / 2        # Add the remaining height of the plate

df

In [15]:
# Calculating the Area
for i, plate in df.iterrows():    # Iterate through the table by row
    area = plate.Size[0] * plate.Size[1]  # Calculate the area value
    df.at[i, '$A=b*d$'] = area            # Stores the area value in the table

df    

In [16]:
# Calculating M
for i, plate in df.iterrows():    # Iterate through the table by row
    M = plate['Distance $y$'] * plate['$A=b*d$']   # Calculate the M value
    df.at[i, '$M=A*y$'] = M
df

In [17]:
# Calculating I_y
for i, plate in df.iterrows():
    I_y = plate['Distance $y$'] * plate['$M=A*y$']   # Calculate the I_y value
    df.at[i, '$I_y=Ay^2$'] = I_y
df

In [18]:
# Calculating I_g
for i, plate in df.iterrows():
    I_g = plate.Size[0] * plate.Size[1] ** 3 / 12   # Calculate the I_g value
    df.at[i, '$I_g=\\frac{bd^3}{12}$'] = I_g
df

In [19]:
# Summing each column
A_sum = df['$A=b*d$'].sum()
A_sum

In [20]:
M_sum = df['$M=A*y$'].sum()
M_sum

In [21]:
I_y_sum = df['$I_y=Ay^2$'].sum()
I_y_sum 

In [22]:
I_g_sum = round(df['$I_g=\\frac{bd^3}{12}$'].sum(), 2)
I_g_sum 

In [23]:
# Getting final answer,
I_n = int(I_y_sum + I_g_sum - M_sum ** 2 / A_sum)  # Cast to int to round off
I_n

and,

In [24]:
n = M_sum / A_sum
n

##### OOP Solution

<div class="warning" style='padding:0.1em; color:Orange; display:inline'>
    <p style='margin-left:1em'>
        In the interest of also allowing this notebook to help teach the reader<br>
        python I've meticulously laid out the development of this class including<br>
        each iteration as it goes through development. Just know that other classes<br>
        won't necessarily be laid out to this detail. The source code of anything<br>
        imported from the civilpy library can be found 
        <a href=https://daneparks.com/Dane/civilpy>here.</a> If you familiarize<br>
        yourself with git, you can see all the various steps that occured in the<br> 
        development of those functions. Git and software development is largely<br>
        outside the scope of this document. So I'll leave those topics to the reader<br>
        to discover.<br><br>
        In OOP, the full cross section would be our "object", so we'd use a class<br>
        to define that and then have the user/application just pass inputs to it<br> 
        as many times as necessary to get the full section. Since a lot of the<br>
        steps in the above calculation are repetitive, we can essentially use the<br>
        class to boil the problem down to inputs and reduce the load on the user<br>
        by providing them a simpler interface that achieves the same result;     
    </p>
</div>

In [25]:
class CrossSection:
    """
    A class for defining a built-up section by repeatedly adding plate 
    dimensions.

    This class allows users to define a built-up cross-section by calling the 
    instance and passing in the dimensions of each plate making up the section. 
    The plates are assumed to be entered in order from the bottom to the top, 
    and all shapes are assumed to be rectangles that do not overlap.

    Note:
        The `shape` parameter is currently unused. It is reserved for a future 
        implementation where non-rectangular shapes may be supported.

    Attributes:
        labels (list of str): A list of labels identifying each plate in the 
            cross-section.
        dimensions (list of tuple): A list of tuples (width, height) for each 
            plate.
        ys (list of float): The centroids of each plate along the y-axis.
        areas (list of float): The areas of each plate.
        moments (list of float): The first moments of area for each plate.
        I_ys (list of float): The second moments of area of each plate about 
            its own centroid.
        I_gs (list of float): The second moments of area of each plate about 
            the global centroid.
        area (float): The total area of the cross-section.
        moment (float): The first moment of area of the cross-section.
        I_y (float): The total second moment of area about the local centroid.
        I_g (float): The total second moment of area about the global centroid.
        I_n (int): The net inertia value, adjusted for global centroid.
        n (float): The location of the neutral axis.

    Args:
        label (str): A label to identify the plate in the cross-section 
            (e.g., "A", "B").
        dimensions (tuple): A tuple representing the plate dimensions 
            (width, height).
        shape (str, optional): The shape of the plate (default is `None` for 
            rectangles).

    Returns:
        CrossSection: An instance of the CrossSection object.
    """
    def __init__(self, label, dimensions, shape=None): 
        self.labels = [label,]
        self.dimensions = [dimensions,]
        (self.ys, 
         self.areas, 
         self.moments, 
         self.I_ys, 
         self.I_gs) = self.calc_properties()
        self.area = sum(self.areas)
        self.moment = sum(self.moments)
        self.I_y = sum(self.I_ys)
        self.I_g = sum(self.I_gs)
        self.I_n = int(self.I_y + self.I_g - (self.moment ** 2 / self.area))
        self.n = self.moment / self.area

    def __call__(self, label, dimensions, shape=None):
        self.labels.append(label)
        self.dimensions.append(dimensions)
        (self.ys, 
         self.areas, 
         self.moments, 
         self.I_ys, 
         self.I_gs) = self.calc_properties()
        self.area = sum(self.areas)
        self.moment = sum(self.moments)
        self.I_y = sum(self.I_ys)
        self.I_g = sum(self.I_gs)
        self.I_n = int(self.I_y + self.I_g - (self.moment ** 2 / self.area))
        self.n = self.moment / self.area

    def __repr__(self):
        return "\n".join([f"{x} {y}" for x, y in zip(
            self.labels, 
            self.dimensions
        )])
        
    def calc_properties(self, shape=None):
        ys = []
        areas = []
        moments = []
        i_ys = []
        i_gs = []
        y = 0
        
        for dimension in self.dimensions:
            # See above functions, these are just reworked versions of those
            y += dimension[1] / 2
            ys.append(y)
            
            area = dimension[0] * dimension[1]
            areas.append(area)
            
            moment = area * y
            moments.append(moment)
            
            i_y = moment * y
            i_ys.append(i_y)

            i_g = dimension[0] * dimension[1] ** 3 / 12
            i_gs.append(i_g)

            y += dimension[1] / 2  # Add the remaining height to 

        return ys, areas, moments, i_ys, i_gs        

<div class="warning" style='padding:0.1em; color:Orange; display:inline'>
    <p style='margin-left:1em'>
        Once the class is defined, the code the user is required to interface<br>
        with becomes much simpler;<br>
        <br>Note that the plates have to be defined from<br>
        bottom to top, like in the practice problem.
    </p>
</div>

In [26]:
test_object = CrossSection('A', (10, 4))
test_object('B', (2, 8))
test_object('C', (6, 4))

In [27]:
test_object

In [28]:
print(f"The value for I_n = {test_object.I_n} in^4")
print(f"and the centroid is at {test_object.n} in.")

<div class="warning" style='padding:0.1em; color:Orange; display:inline'>
    <p style='margin-left:1em'>
        Here we used the final values defined in the class object of <code>self.I_n</code><br> 
        and <code>self.n</code> to print the same results given in the original book, but<br>
        all of the intermediate values we defined while calculating those values<br>
        are also available for checking, just not in as clean of a tabular<br>
        format. <code>self.area</code>, <code>self.moment</code>, <code>self.I_y</code>, <code>self.I_g</code>, <code>self.areas</code>,<br>
        <code>self.moments</code>, <code>self.I_ys</code>, or <code>self.I_gs</code> could all be printed out while<br>
        running this notebook to verify that the application is running correctly<br>
        during intermediary steps;<br><br>
        In python <code>self</code> refers to the object you're currently defining, when using<br>
        the object, replace self with the object you defined with the class, in this<br>
        case, <code>test_object</code>.
    </p>
</div>

In [29]:
test_object.areas, sum(test_object.areas), test_object.area

In [30]:
# Update the class to allow adding sections at an arbitrary height;
class CrossSection:
    """
    A class for defining a built-up section by repeatedly adding plate 
    dimensions.

    This class allows users to define a built-up cross-section by calling the 
    instance and passing in the dimensions of each plate making up the section. 
    The plates are assumed to be entered in order from the bottom to the top, 
    and all shapes are assumed to be rectangles that do not overlap.

    Note:
        The `shape` parameter is currently unused. It is reserved for a future 
        implementation where non-rectangular shapes may be supported.

    Attributes:
        labels (list of str): A list of labels identifying each plate in the 
            cross-section.
        dimensions (list of tuple): A list of tuples (width, height) for each 
            plate.
        ys (list of float): The centroids of each plate along the y-axis.
        areas (list of float): The areas of each plate.
        moments (list of float): The first moments of area for each plate.
        I_ys (list of float): The second moments of area of each plate about 
            its own centroid.
        I_gs (list of float): The second moments of area of each plate about 
            the global centroid.
        area (float): The total area of the cross-section.
        moment (float): The first moment of area of the cross-section.
        I_y (float): The total second moment of area about the local centroid.
        I_g (float): The total second moment of area about the global centroid.
        I_n (int): The net inertia value, adjusted for global centroid.
        n (float): The location of the neutral axis.

    Args:
        label (str): A label to identify the plate in the cross-section 
            (e.g., "A", "B").
        dimensions (tuple): A tuple representing the plate dimensions 
            (width, height).
        shape (str, optional): The shape of the plate (default is `None` for 
            rectangles).

    Returns:
        CrossSection: An instance of the CrossSection object.
    """
    def __init__(self, label, dimensions, shape=None):
        self.labels = [label, ]
        self.dimensions = [dimensions, ]
        (self.ys,
         self.areas,
         self.moments,
         self.I_ys,
         self.I_gs) = self.calc_properties()
        self.area = sum(self.areas)
        self.moment = sum(self.moments)
        self.I_y = sum(self.I_ys)
        self.I_g = sum(self.I_gs)
        self.I_n = int(self.I_y + self.I_g - (self.moment ** 2 / self.area))
        self.n = self.moment / self.area

    def __call__(self, label, dimensions, y=None, shape=None):
        self.labels.append(label)
        self.dimensions.append(dimensions)
        if y is None:
            (self.ys,
             self.areas,
             self.moments,
             self.I_ys,
             self.I_gs) = self.calc_properties()
        else:
            self.append_value(y)
        self.area = sum(self.areas)
        self.moment = sum(self.moments)
        self.I_y = sum(self.I_ys)
        self.I_g = sum(self.I_gs)
        self.I_n = int(self.I_y + self.I_g - (self.moment ** 2 / self.area))
        self.n = round(self.moment / self.area, 2)

    def __repr__(self):
        return "\n".join([f"{x} {y}" for x, y in zip(
            self.labels,
            self.dimensions
        )])

    def calc_properties(self, y=None, shape=None):
        ys = []
        areas = []
        moments = []
        i_ys = []
        i_gs = []
        y = 0

        for dimension in self.dimensions:
            # See above functions, these are just reworked versions of those
            y += dimension[1] / 2
            ys.append(y)

            area = dimension[0] * dimension[1]
            areas.append(area)

            moment = area * y
            moments.append(moment)

            i_y = moment * y
            i_ys.append(i_y)

            i_g = dimension[0] * dimension[1] ** 3 / 12
            i_gs.append(i_g)

            y += dimension[1] / 2  # Add the remaining height to

        return ys, areas, moments, i_ys, i_gs

    def append_value(self, y, shape=None):
        self.ys.append(y)
        self.areas.append(self.dimensions[-1][0] * self.dimensions[-1][1])
        self.moments.append(self.areas[-1] * self.ys[-1])
        self.I_ys.append(self.moments[-1] * self.ys[-1])
        if shape is None:
            self.I_gs.append(
                (self.dimensions[-1][0] * self.dimensions[-1][1] ** 3)
                / 12)
        else:
            pass

In [31]:
# Recreate the test_object with the new definition
test_object = CrossSection('A', (10, 4))
test_object('B', (2, 8))
test_object('C', (6, 4))

In [32]:
test_object.I_n

In [33]:
test_object.n

In [34]:
# This can normally be used when the shape isn't historic
# from civilpy.structural.steel import W
# w18x97 = W("W18x97")  # The shape used in the book though is historical

In [35]:
from civilpy.structural.steel import WF

Update the `CrossSection` class to be able to incorporate rolled steel shapes, at an  
arbitrary height `y`,

In [36]:
class CrossSection:
    """
    A class for defining a built-up section by repeatedly adding plate 
    dimensions.

    This class allows users to define a built-up cross-section by calling the 
    instance and passing in the dimensions of each plate making up the section. 
    The plates are assumed to be entered in order from the bottom to the top, 
    and all shapes are assumed to be rectangles that do not overlap.

    Attributes:
        labels (list of str): A list of labels identifying each plate in the 
            cross-section. 
        dimensions (list of tuple): A list of tuples (width, height) for each 
            plate.
        ys (list of float): The centroids of each plate along the y-axis.
        areas (list of float): The areas of each plate.
        moments (list of float): The first moments of area for each plate.
        I_ys (list of float): The second moments of area of each plate about 
            its own centroid.
        I_gs (list of float): The second moments of area of each plate about 
            the global centroid.
        area (float): The total area of the cross-section.
        moment (float): The first moment of area of the cross-section.
        I_y (float): The total second moment of area about the local centroid.
        I_g (float): The total second moment of area about the global centroid.
        I_n (int): The net inertia value, adjusted for global centroid.
        n (float): The location of the neutral axis.

    Args:
        label (str): A label to identify the plate in the cross-section (e.g., 
            "A", "B").
        dimensions (tuple): A tuple representing the plate dimensions (width, 
            height).
        shape (str, optional): The shape of the plate (default is `None` for 
            rectangles).

    Returns:
        CrossSection: An instance of the CrossSection object.
    """
    def __init__(self, label, dimensions=None, shape=None, y=None, 
                 axis='strong'):
        self.labels = [label, ]
        if shape:
            self.shape = shape  # //TODO - Replace with generalized function
            self.dimensions = [(
                self.shape.flange_width.magnitude,
                self.shape.depth.magnitude
            ),]
            self.areas = [float(self.shape.area.magnitude),]
            self.I_gs = [float(self.shape.I_x.magnitude),]
        else:
            self.dimensions = [dimensions,]
            self.areas = [dimensions[0]*dimensions[1],]
            self.I_gs = [dimensions[0] * dimensions[1] ** 3 / 12,]

        if not y:
            y = self.dimensions[0][1] / 2
            self.ys = [y,]
            
        self.ys = [y,]
        self.moments = [self.areas[0] * y,]
        self.I_ys = [self.moments[0] * y,]
        self.height = self.dimensions[0][1] / 2 + y
        self._calc_gen_properties()
        

    def __call__(self, label, dimensions=None, y=None, shape=None, 
                 axis='strong'):
        self.labels.append(label)
        if shape:
            self.shape = shape
        self.append_value(dimensions, y, shape)

    def __repr__(self):
        return "\n".join([f"{x} {y}" for x, y in zip(
            self.labels,
            self.dimensions
        )])

    def append_value(self, dimensions=None, y=None, shape=None, 
                     axis=None):
        if y is None and shape is None:  # Adding rect sect at top of xsection
            self.dimensions.append(dimensions)
            y = self.height + dimensions[1] / 2
            self.height = self.height + dimensions[1]
            area=self.dimensions[-1][0] * self.dimensions[-1][1]
            self.I_gs.append(
                (self.dimensions[-1][0] * self.dimensions[-1][1] ** 3) / 12)
        
        elif shape and y:  # Shape provided, with y value
            area=float(self.shape.area.magnitude)
            if axis=='strong':
                self.dimensions.append((
                    self.shape.flange_width.magnitude,
                    self.shape.depth.magnitude))
                self.height += self.shape.depth.magnitude / 2 + y
                self.I_gs.append(float(self.shape.I_x.magnitude))
            else:
                self.dimensions.append((
                    self.shape.depth.magnitude,
                    self.shape.flange_width.magnitude
                ))
                self.height += self.shape.flange_width.magnitude / 2 + y
                self.I_gs.append(float(self.shape.I_y.magnitude))
        
        elif shape and y is None:  # Shape provided, no y value
            # //TODO - Update to account for non-symmetrical shapes
            # //TODO - Update to use a general function
            if axis=='strong':
                self.dimensions.append((
                    self.shape.flange_width.magnitude,
                    self.shape.depth.magnitude))
            else:  
                self.dimensions.append((
                    self.shape.depth.magnitude,
                    self.shape.flange_width.magnitude
                    ))
            y = self.height + (self.shape.depth.magnitude / 2)
            self.height += self.shape.depth.magnitude
            area=float(self.shape.area.magnitude)
            self.I_gs.append(float(self.shape.I_x.magnitude))

        elif shape is None and y:  # Rectangular box at y height
            self.dimensions.append(dimensions)
            area=self.dimensions[-1][0] * self.dimensions[-1][1]
            self.I_gs.append(
                round((self.dimensions[-1][0] * 
                       self.dimensions[-1][1] ** 3) / 
                      12, 2))
        else:
            print("Unexpected execution")
        self.areas.append(area)
        self.ys.append(float(y))
        self.moments.append(self.areas[-1] * self.ys[-1])
        self.I_ys.append(self.moments[-1] * self.ys[-1])
        
        self._calc_gen_properties()
    # //TODO - Update a function in the steel library to accept general input
    # //TODO - Update Steel library to ID symmetrical shapes (no y value)

    def check_negative_y_values(self):
        for value in self.plate_dims.keys():
            if value < 0:
                return True
        return False
    
    def _calc_gen_properties(self):
        self.area = sum(self.areas)
        self.moment = sum(self.moments)
        self.I_y = sum(self.I_ys)
        self.I_g = sum(self.I_gs)
        self.I_n = round(
            self.I_y + self.I_g - (self.moment ** 2 / self.area), 
            1)
        self.n = round(self.moment / self.area, 3)
        self.plate_dims = {x: y[1] for x, y in zip(self.ys, self.dimensions)}
        # //TODO - have this check both negative and positive
        plate_dims = {x: y[1] for x, y in zip(self.ys, self.dimensions)}
        extr_neg_y = min(plate_dims.keys()) - \
                plate_dims[min(plate_dims.keys())] / 2

        if self.check_negative_y_values():
            self.cb = extr_neg_y - self.n
        else:
            self.cb = self.height - self.n
        self.S = round(self.I_n / self.cb, 0)

In [37]:
w18x96 = WF('18WF96', '18WF_B18b')
w18x96.area, w18x96.I_y, w18x96.web_thickness

In [38]:
# Verify we didn't break our class with these updates
test_object = CrossSection('A', shape=w18x96, y=5)
test_object.height

In [39]:
# Recreate the test_object with the new definition
test_object = CrossSection('A', (10, 4))
test_object('B', (2, 8))
test_object('C', (6, 4))
test_object.I_n

In [40]:
test_object.n

In [41]:
test_object('D', (2, 4), 14)
test_object.I_n

In [42]:
test_object.n

In [43]:
# Attempt book problem #5
test_object = CrossSection('A', (16, 2), y=-17)
test_object('B', (1, 32))
test_object('C', shape=w18x96, y=16.256, axis='weak')

In [44]:
test_object.I_n

In [45]:
test_object.n

In [46]:
test_object.cb

In [47]:
test_object.S

<div class="warning" style='padding:0.1em; color:Orange; display:inline'>
    <p style='margin-left:1em'>
        For sections more complicated than this, it is recommended to start utilizing<br>
        a dedicated python library like <a href='https://sectionproperties.readthedocs.io/en/stable/'>SectionProperties.</a> This will greatly<br>
        simplify the process as it'll allow you to start drawing shapes in CADD rather<br>
        than requiring you to perform extensive calculations or writing intricate python<br>
        classes to account for every scenario.<br><br>
        Note the use of going back and using the previously defined functions to <br>
        verify changes to the class didn't break it's functionality. These functions<br>
        called "tests" are often defined prior to ever writing the class. This is a<br>
        process called test driven development or "TDD". It is a good practice to get<br>
        in the habit of but not absolutely essential. Good testing will allow you to<br>
        verify your code still works, despite making potentially massive changes<br>
        to your codebase. There are my styles and types of testing, like unit tests<br>
        functional testing and docstring tests. Again, testing could be a book in and<br>
        of itself, so it's up to the reader to delve into that topic further.
    </p>
</div>

In [48]:
test_object = CrossSection('A', (1.5, 15), y=7.5)
test_object('B', (6, 1.5), y=.75)

In [49]:
test_object.n

In [50]:
test_object.I_n

In [51]:
test_object.S

In [56]:
import math
r = round(math.sqrt(test_object.I_n / test_object.area), 2)
r